# 单通道：池化 trace 的闪烁特性

输入只有 `smTest_manual` 池化出来的 `TRACE_pooled/<组名>/traces_pooled.npz`，
**不碰影像**：不重新提取、不重新扣背景，只处理 trace 本身。

| 阶段 | 放在哪里 |
|---|---|
| 取 trace、人工挑、池化 | `smTest_batch` → `smTest_manual` |
| **闪烁统计（这一本）** | 只吃 `traces_pooled.npz` |

两个必须记住的事实：

1. **尾部 NaN**：池化为了对齐帧数，短的 trace 在后面补了 NaN。带着 NaN 跑
   `changepoint.segment` **不会报错**，但 `cumsum` 被污染，整条会被判成
   "一个暗段"（实测 139 条全部 `n_states=1`）。所以下面读盘时先按 `n_frames` 截断。
2. **每条的帧数不同**（1000 / 2000），任何统计都按条处理，不能当成 `(N, T)`
   整齐矩阵用。

闪烁分析用到的函数（`changepoint.*`、`summarize_states`、`runs_of`、
`select_traces`、`plot_state_summary`）都只读 trace 自己的字段，不需要影像；
`ThresholdStates` / `ChangepointStates` 的 `fill(trace, ctx, box)` 也**不碰**
`ctx` / `box`，所以 `fill(trace, None, None)` 就能对读回来的 trace 直接判状态。


In [ ]:
# `import smlm` 需要 Py_Smile 在 sys.path 上; notebook 放在子目录里时自己找一下
import pathlib
import sys

for _q in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_q / 'smlm' / '__init__.py').is_file():
        sys.path.insert(0, str(_q))
        break

import matplotlib.pyplot as plt
import numpy as np

import smlm.trace as tr

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ======================= 输入 =======================
BASE_DIR = pathlib.Path(r"G:\SPARK\20261124")
POOL_DIR = BASE_DIR / 'TRACE_pooled'
GROUP = 'SPARK_6mW'                    # 先做这一组; 换组只改这里
POOLED = POOL_DIR / GROUP / 'traces_pooled.npz'

print('输入 :', POOLED)
print('存在 :', POOLED.is_file())


In [ ]:
def load_traces(path):
    """读 traces_pooled.npz -> (traces, files, dt, n_columns)。

    按 ``n_frames`` 截掉尾部的 NaN —— 否则变点检测会静默退化成"整条一个暗段"。
    """
    with np.load(path) as z:
        I = np.asarray(z['intensity'], np.float32)
        B = np.asarray(z['bg_flux'], np.float32)
        ids = np.asarray(z['emitter_id'], int)
        fid = np.asarray(z['file_id'], int)
        files = [str(s) for s in z['files']]
        nf = np.asarray(z['n_frames'], int)
        t_s = np.asarray(z['t_s'], float)
        dt = float(z['exposure_s'])
    traces = []
    for i in range(len(ids)):
        n = int(nf[i])
        traces.append(dict(emitter_id=int(ids[i]), file_id=int(fid[i]),
                           file=files[fid[i]], frames=np.arange(n),
                           t_s=t_s[:n], intensity=I[i, :n],
                           bg_flux=B[i, :n]))
    return traces, files, dt, I.shape[1]


traces, files, dt, n_col = load_traces(POOLED)

n_frame = np.array([len(t['frames']) for t in traces])
peak = np.array([np.percentile(t['intensity'], 95) for t in traces])
em = np.array([t['emitter_id'] for t in traces])
fid = np.array([t['file_id'] for t in traces])

print(f'\n曝光 {dt:g} s, 池化数组宽 {n_col} 帧 (= 最长的一条, 短的补 NaN)')
print(f'共 {len(traces)} 条 trace, 帧数 {n_frame.min()}~{n_frame.max()}')
for k, name in enumerate(files):
    m = fid == k
    print(f'  [{k}] {name:<44} {m.sum():>4} 条, '
          f'{n_frame[m].max() if m.any() else 0} 帧')
print(f'\n95% 分位亮度: 中位 {np.median(peak):.0f}, '
      f'范围 {peak.min():.0f}~{peak.max():.0f}')
print('前 3 条:', [int(v) for v in em[:3]])


## 先看一眼数据

按 95% 分位亮度降序（先看亮的）。左列是扣背景后的信号 `intensity`，右列是
`bg_flux`（孔径里的背景光子数，用来判断视野有没有被背景淹没）。


In [ ]:
order = np.argsort(-peak)[:12]

fig = tr.plot_trace_fields(
    traces, indices=order,
    groups={'intensity': ('intensity',),
            'bg_flux': dict(fields=('bg_flux',), color='tab:red')},
    xlabel=f'Frame  ({dt:g} s/frame)',
    title=f'{GROUP}   最亮的 {len(order)} 条')
plt.show()


一屏看完所有 trace：按亮度排序的 `intensity` 热图（横轴帧、纵轴 trace）。
帧数不同的那些在右边是**空白**，不是 0。图存到
`TRACE_pooled/<组名>/blink_heatmap.png`。


In [ ]:
def heatmap(traces, title=None, figsize=(10, 4)):
    """按亮度排序的一屏热图; 帧数不同 -> 短的在右边留白。"""
    order = np.argsort([-np.percentile(t['intensity'], 95) for t in traces])
    T = max(len(t['intensity']) for t in traces)
    M = np.full((len(traces), T), np.nan)
    for r, i in enumerate(order):
        M[r, :len(traces[i]['intensity'])] = traces[i]['intensity']
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(M, aspect='auto', cmap='magma', origin='lower',
                   interpolation='nearest')
    ax.set_xlabel('Frame')
    ax.set_ylabel('trace (sorted by brightness)')
    ax.set_title(title or '')
    fig.colorbar(im, ax=ax, label='intensity')
    fig.tight_layout()
    return fig


heatmap(traces, title=f'{GROUP}   n={len(traces)}').savefig(
    POOLED.parent / 'blink_heatmap.png', dpi=150,
    bbox_inches='tight', facecolor='white')
plt.show()


## 下一步：闪烁特性（工具还在等你拍板）

状态判定不需要影像，直接这样跑就有 `state` / `n_on` / `n_switches` / `bleach_frame`：

```python
filler = tr.ChangepointStates(bleach_frames=500, vst=None)
for t in traces:
    filler.fill(t, None, None)      # ctx / box 这两格它不用
tr.plot_state_summary(traces, title=GROUP)
```

再往下的 ON/OFF 段长与拟合还没做，要定三件事：

| 待定 | 备选 |
|---|---|
| `bleach_frames` | 1000 帧的视野（14 条）用 500 会不会太长 |
| `vst` | 这组 95% 分位中位只有 49（暗），`'anscombe'` 可能更好；亮分子反而 raw 更好 |
| OFF 段怎么取 | 只从**存活区间**取（漂白后的长暗段不能算闪烁），首尾截断是否做校正 |
